### mediapipe 사용 Pose Estimation (전신 33개 관절 추적)  예제

### MediaPipe
구글이 만든 멀티모달 머신러닝 프레임워크 <br>
특히 영상이나 이미지에서 사람 손, 얼굴, 포즈 같은 걸 탐지하고 추적하는 데 강력하다. <br>
#### 주요 특징: 
Pre-built 모델을 제공한다 (ex. 얼굴 검출, 손 추적, 전체 포즈 분석 등) <br>
실시간 처리가 가능 (모바일/PC 모두) <br>
C++와 Python 둘 다 지원 <br>
TensorFlow Lite 기반으로 모바일 최적화가 잘 되어 있다<br>
Graph 구조로 처리 파이프라인을 구성한다 (필터, 모델 호출, 후처리 등을 자유롭게 연결 가능) <br>

#### 지원 기능 :
- Face Detection (얼굴 검출) <br>
- Face Mesh (얼굴 랜드마크 468개 포인트 추적)  <br>
- Hand Tracking (손가락 21개 관절 포인트 추적) <br>
- Pose Estimation (전신 관절 33개 추적) <br>
- Objectron (3D 객체 인식)

In [ ]:
# ! pip install mediapipe opencv-python numpy

In [ ]:
# import sys
# !{sys.executable} -m pip uninstall tensorflow tensorflow-intel -y
# !{sys.executable} -m pip install mediapipe==0.10.9 "protobuf>=3.20.3,<5"

In [1]:
import cv2
import mediapipe as mp

# MediaPipe Pose 모듈 불러오기
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

# 랜드마크 그리기용
mp_drawing = mp.solutions.drawing_utils

# 웹캠 열기
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # BGR -> RGB 변환 (MediaPipe는 RGB 사용)
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # 이미지 쓰기 방지 설정 해제 (성능 최적화용)
    image.flags.writeable = False

    # Pose 모델에 이미지 전달 , 사람 포즈를 분석
    results = pose.process(image)

    # 다시 이미지 쓰기 가능 설정
    image.flags.writeable = True

    # RGB -> BGR 변환 (OpenCV 출력용)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # 랜드마크가 있으면 그리기
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            image,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2)
        )

    # 결과 화면 보여주기
    cv2.imshow('MediaPipe Pose Estimation', image)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### 관절 좌표 (x, y, z)를 화면에 같이 표시

landmark.x, landmark.y, landmark.z는 각각 0~1 사이의 상대좌표, z는 깊이 방향

In [3]:
import cv2
import mediapipe as mp

# MediaPipe Pose 모듈 불러오기
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

# 랜드마크 그리기용
mp_drawing = mp.solutions.drawing_utils

# 웹캠 열기
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # BGR -> RGB 변환 (MediaPipe는 RGB 사용)
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    image.flags.writeable = False
    results = pose.process(image)   #  사람 포즈를 분석
    image.flags.writeable = True

    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    if results.pose_landmarks:
        # 랜드마크 그리기
        mp_drawing.draw_landmarks(
            image,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2)
        )

        # 랜드마크 좌표 가져오기 : 33개 관절 하나하나 순회
        for idx, landmark in enumerate(results.pose_landmarks.landmark):
            h, w, _ = image.shape
            cx, cy = int(landmark.x * w), int(landmark.y * h)

            # (idx) (x,y,z) 값을 이미지에 텍스트로 출력
            cv2.putText(
                image,
                f'{idx}',
                (cx, cy),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4,
                (255, 255, 255),
                1,
                cv2.LINE_AA
            )

    cv2.imshow('MediaPipe Pose Estimation with Landmarks', image)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()